Layer: Transformation → Silver
Source: Bronze (Delta Tables)
Target: Cleaned Silver Table


 1 Read Bronze Table


In [0]:
df_bronze_sales_orders = spark.read.table("workspace.bronze.erp_sales_orders") 

In [0]:
# Preview data
display(df_bronze_sales_orders)

# Check schema
df_bronze_sales_orders.printSchema()

In [0]:
df_bronze_sales_orders.columns


 NOTE:
 Column naming follows best practices:
 - snake_case format
 - lowercase letters only
 - descriptive and explicit names
   e.g., order_id, customer_id, unit_price


 2 Data Type Casting



In [0]:
import pyspark.sql.functions as F

df_typed = df_bronze_sales_orders\
    .withColumn("order_id", F.col("order_id").cast("string")) \
    .withColumn("customer_id", F.col("customer_id").cast("string")) \
    .withColumn("product_id", F.col("product_id").cast("string")) \
    .withColumn("order_date", F.to_date(F.col("order_date"), "dd/MM/yyyy")) \
    .withColumn("quantity", F.col("quantity").cast("int")) \
    .withColumn("unit_price", F.col("unit_price").cast("double")) \
    .withColumn("total_amount", F.col("total_amount").cast("double")) \
    .withColumn("status", F.col("status").cast("string")) \
    .withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("timestamp")) \
    .withColumn("source_file", F.col("source_file").cast("string"))

In [0]:
df_typed.printSchema()

In [0]:
from pyspark.sql.functions import regexp_replace, col

# product_id (already fine)
df_typed = df_typed.withColumn(
    "product_id",
    regexp_replace(col("product_id"), "^PROD", "")
)

# customer_id: remove prefix + leading zeros
df_typed = df_typed.withColumn(
    "customer_id",
    regexp_replace(
        regexp_replace(col("customer_id"), "^CUST", ""),
        "^0+",
        ""
    )
)

In [0]:
df_typed.select("product_id", "customer_id").show(5)


 3 Handle Null Values



 DATA QUALITY CONSTRAINTS (ERP - SALES ORDERS)


 Business rules applied in Silver layer:
#
 - order_id must be unique
 - customer_id cannot be NULL
 - product_id cannot be NULL
 - quantity >= 0
 - unit_price >= 0
 - total_amount >= 0
 - order_date must be a valid date
 - status must be one of: (Pending, Completed, Cancelled)


In [0]:
import pyspark.sql.functions as F

df_cleaned = df_typed \
    .dropna(subset=["order_id", "customer_id", "order_date"]) \
    .fillna({
        "quantity": 0,
        "unit_price": 0,
        "total_amount": 0
    })

In [0]:


df_cleaned.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["order_id", "customer_id", "order_date"]
]).show()

In [0]:
from pyspark.sql import functions as F

df_cleaned.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["quantity", "unit_price", "total_amount"]
]).show()


 4 Deduplication 


In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

window_spec = Window.partitionBy("order_id").orderBy(F.col("ingestion_timestamp").desc())

df_deduplicated = df_cleaned \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

In [0]:
#Validation
df_deduplicated.groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


 6 Data Quality Rules Enforcement


In [0]:
from pyspark.sql import functions as F

df_deduplicated.filter(
    (F.col("quantity") < 0) |
    (F.col("unit_price") < 0) |
    (F.col("total_amount") < 0) |
    (F.col("status").isin("Pending", "Completed", "Cancelled"))
).display()

In [0]:
df_valid = df_deduplicated.filter(
    (F.col("quantity") >= 0) &
    (F.col("unit_price") >= 0) &
    (F.col("total_amount") >= 0) &
    (F.col("status").isin("Pending", "Completed", "Cancelled"))
)

 7 Derived Columns 


In [0]:
from pyspark.sql import functions as F

df_enriched = df_valid \
    .withColumn("total_amount", F.col("quantity") * F.col("unit_price")) \
    .withColumn("order_year", F.year(F.col("order_date"))) \
    .withColumn("order_month", F.month(F.col("order_date")))

In [0]:
#Validation
df_enriched.select(
    "quantity",
    "unit_price",
    "total_amount",
    "order_year",
    "order_month"
).display(5)

In [0]:
df_enriched = df_enriched.withColumn(
    "product_id",
    regexp_replace(col("product_id"), "^0+", "")
)


 8 Write to Silver Table


In [0]:
spark.sql("DROP TABLE IF EXISTS silver.sales_orders")

In [0]:
df_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.sales_orders")

In [0]:
spark.read.table("silver.sales_orders").count()

In [0]:
display(spark.table("silver.sales_orders"))

====================================================
✅ SILVER LAYER VALIDATION CHECKLIST
============================================

### Schema Validation

In [0]:
spark.read.table("silver.sales_orders").printSchema()

### 2. Row Count Consistency

In [0]:
df_enriched.count(), spark.read.table("silver.sales_orders").count()

### 3. Uniqueness Check

In [0]:
from pyspark.sql import functions as F

spark.read.table("silver.sales_orders") \
    .groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

### 4. Null Check

In [0]:
spark.read.table("silver.sales_orders").select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["order_id", "customer_id", "order_date"]
]).show()

### 5. Numeric Validation

In [0]:
spark.read.table("silver.sales_orders").filter(
    (F.col("quantity") < 0) |
    (F.col("unit_price") < 0) |
    (F.col("total_amount") < 0)
).show()

### 6. Business Rule Check

In [0]:
spark.read.table("silver.sales_orders") \
    .select("status").distinct().show()

### 7. Derived Columns Check

In [0]:
spark.read.table("silver.sales_orders").select(
    "quantity", "unit_price", "total_amount"
).show(5)

### 8. Incremental Behavior Check

In [0]:
spark.read.table("silver.sales_orders").count()

In [0]:
df_enriched.select("product_id", "customer_id").show(10)


In [0]:
spark.table("silver.sales_orders") \
    .join(spark.table("silver.products"), "product_id") \
    .count()
    